In [ ]:
import requests
!pip install ImageHash
import imagehash
from PIL import Image
from io import BytesIO
import json
!pip install tqdm
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time

# Scryfall allows up to 10 requests per second.
RATE_LIMIT = 10  # requests per second
rate_limiter_lock = threading.Lock()
last_request_time = [0.0]  # Use list for mutability in threads

def scryfall_rate_limiter():
    with rate_limiter_lock:
        now = time.time()
        elapsed = now - last_request_time[0]
        min_interval = 1.0 / RATE_LIMIT
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)
        last_request_time[0] = time.time()

# Step 1: Download Scryfall bulk card data (Default Cards)
bulk_data_url = "https://api.scryfall.com/bulk-data"
scryfall_rate_limiter()
bulk_data_response = requests.get(bulk_data_url)
if bulk_data_response.status_code == 429:
    print("Rate limited when accessing bulk data endpoint.")
bulk_data = bulk_data_response.json()

# Find the "Default Cards" download link
default_cards_url = next(entry['download_uri'] for entry in bulk_data['data'] if entry['type'] == 'default_cards')

# Download the actual card data
scryfall_rate_limiter()
cards_data_response = requests.get(default_cards_url)
if cards_data_response.status_code == 429:
    print("Rate limited when downloading card data.")
cards_data = cards_data_response.json()

# Step 2: Process each card and compute phash in parallel
def process_card(card):
    if 'image_uris' not in card:
        return None  # Skip double-faced cards or tokens without image_uris
    if card.get('lang') != 'en':
        return None  # Only process English cards
    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            image_url = card['image_uris']['normal']
            scryfall_rate_limiter()
            image_response = requests.get(image_url, timeout=10)
            if image_response.status_code == 429:
                print(f"Rate limited when downloading image for card: {card.get('name')}")
                return None
            image_response.raise_for_status()
            img = Image.open(BytesIO(image_response.content)).convert("RGB")
            img_hash = str(imagehash.phash(img))
            return {
                "phash": img_hash, # The image hash (important actually)
                "scryfall_uri": card['scryfall_uri'], # Adds the URL to the card on scryfall
                "name": card['name'], # Adds card name field (important actually)
                "set": card['set'], # Adds set name field
                "mana_cost": card.get('mana_cost')  # Adds mana cost field (important actually)
            }
        except Exception as e:
            if attempt == max_attempts - 1:
                print(f"Failed to process {card.get('name')}: {e}")
                return None

hash_db = []
with ThreadPoolExecutor(max_workers=25) as executor:
    futures = [executor.submit(process_card, card) for card in cards_data]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Hashing cards"):
        result = f.result()
        if result:
            hash_db.append(result)

# Step 3: Save the hash database to a JSON file
output_path = "scryfall_phash_db_sample.json"
with open(output_path, "w") as f:
    json.dump(hash_db, f, indent=2)

output_path  # Provide download link to user